# Phase 28 — the soft-onset gate

**This is a GATE, not a training run.** It spends ~1 GPU hour to decide whether a
full run is worth ~1 GPU week. Three short runs at different boost values, plus
an unboosted control, all from the same checkpoint and the same seed.

## Why it exists

This project's last two training efforts were full-cost negatives that a small
probe would have caught:

| | outcome |
|---|---|
| Phase 23, 10,000 steps, `--loss-weights velocity=0.1` | **−0.0050 onset F1**, precision 0.8753 → 0.8657 |
| Phase 25 per-beat floor | +0.0182 on 5 tempi, **−0.0082 on 9** |

## What is being tested

Phase 27 measured the onset head missing **38.3% of pp notes against 2.4% of f
notes** — a 16x spread, `pp`+`p` being 66.6% of all missed notes from 33.4% of
the reference. `--soft-onset-boost` weights each note's onset loss by how quiet
it is. Nothing else in the stack can express that: `--loss-weights` scales four
HEADS by four scalars.

**Not a repeat of Phase 23.** That downweighted the VELOCITY head — how hard the
model tries to predict loudness. This changes how hard it is pushed to DETECT
quiet notes, in the ONSET head.

**The cheap alternative is already eliminated.** A velocity-aware decode
threshold was measured and lands exactly on the global precision/recall curve
(`benchmarks/velocity-aware-decode-README.md`) — the soft-note activations are
not there to recover, so training is the only remaining path.

## The gate criterion — decide this BEFORE reading the numbers

A boost passes if, against the **boost=1.0 control at the same step count**:

1. `val_onset` is **not worse** — the run has not simply destabilised, and
2. the trend is visible by 1,500 steps rather than requiring faith in step 10,000.

**A pass here does NOT mean the idea works.** 1,500 steps cannot show a real F1
gain. It means the mechanism trains stably and is worth the full run. The full
run is where the actual question — does soft-note recall rise while precision
holds — gets answered against `maps-paired-ptify16b-at060.json` (**0.8502**).

In [ ]:
COMMIT = "phase-27-recall-diagnosis"   # branch, tag, or full SHA
REPO = "https://github.com/ImSe4n/PTify.git"

# --no-deps: Kaggle's torch is far newer than this project's local pin, which is
# fine (a checkpoint is a plain state_dict), but letting a resolver reinstall
# torch would burn most of the session. The cost is naming transitive imports.
!pip install -q --no-deps git+{REPO}@{COMMIT}
!pip install -q --no-deps piano_transcription_inference torchlibrosa \
    mido pretty_midi librosa soundfile audioread soxr lazy_loader \
    msgpack mir_eval

In [ ]:
!git clone -q --depth 1 --branch {COMMIT} {REPO} /kaggle/working/PTify

import glob, os
print("--- attached datasets ---")
for path in sorted(glob.glob("/kaggle/input/*/")):
    print(path)
    for sub in sorted(glob.glob(path + "*"))[:6]:
        print("   ", os.path.basename(sub))

In [ ]:
# The directory must CONTAIN the year folders (2004/ ... 2018/): the segment
# index stores relative paths like `2011/MIDI-Unprocessed_....wav`.
AUDIO_ROOT = "/kaggle/input/the-maestro-dataset-v3-0-0/maestro-v3.0.0"

# The 16b weights this gate starts from. Every arm continues the SAME
# checkpoint, so the only difference between arms is the boost.
INIT_CKPT = "/kaggle/input/ptify-16b/ptify-16b-step6555.pth"

import json, pathlib
index = json.load(open("/kaggle/working/PTify/benchmarks/maestro_segments.json"))
probe = pathlib.Path(AUDIO_ROOT) / index["tracks"][0]["audio_filename"]
print("probe :", probe)
print("EXISTS:", probe.exists(), "<- must be True")
print("init  :", pathlib.Path(INIT_CKPT).exists(), "<- must be True")

## The four arms

`1.0` is the **control** and is not optional. Without it, every number below is
uninterpretable: a validation loss means nothing except against the same
recipe's own unboosted run at the same step. Phase 23's phantom `+0.0057` came
from comparing against a baseline measured under different settings, and this
cell exists so that cannot happen here.

Three boosts rather than one because **a single value chosen by intuition is how
the Phase 24 rate floor got tuned at one tempo and looked perfect until it was
measured at nine**. 2/4/8 spans a factor of four; if the effect is real, its
direction should be visible across them.

Everything else is held fixed: same init, same seed, same augmentation, same
step count.

In [ ]:
STEPS = 1500
ARMS = [1.0, 2.0, 4.0, 8.0]   # 1.0 FIRST: the control, and the fastest failure

import subprocess, time

for boost in ARMS:
    tag = f"boost{boost:g}"
    out = f"/kaggle/working/gate/{tag}"
    print(f"\n=== {tag} ===", flush=True)
    t0 = time.time()
    rc = subprocess.call([
        "python", "-m", "training.train",
        "--index", "benchmarks/maestro_segments.json",
        "--audio-root", AUDIO_ROOT,
        "--out", out,
        "--init-checkpoint", INIT_CKPT,
        "--soft-onset-boost", str(boost),
        "--augment", "--augment-seed", "0", "--augment-clean-prob", "0.10",
        "--device", "cuda", "--no-amp",
        "--steps", str(STEPS),
        "--batch-size", "4", "--accum-steps", "2", "--workers", "2",
        "--log-every", "50",
        "--validate-every", "250", "--val-batches", "20",
        "--save-every-seconds", "1800",
        "--keep-checkpoints", "1",
        "--resume", "auto",
    ], cwd="/kaggle/working/PTify")
    print(f"{tag}: rc={rc} in {(time.time()-t0)/60:.1f} min", flush=True)

In [ ]:
# Validation is scored UNWEIGHTED in every arm (`evaluate` drops onset_weight),
# so these curves are directly comparable. That is deliberate and tested --
# without it each arm's val_onset would sit at a different level purely because
# of its boost, and the comparison would be meaningless.
import json, pathlib

print(f"{'arm':<10} {'step':>6} {'val_total':>10} {'val_onset':>10}")
summary = {}
for boost in ARMS:
    tag = f"boost{boost:g}"
    log = pathlib.Path(f"/kaggle/working/gate/{tag}/train_log.jsonl")
    if not log.exists():
        print(f"{tag:<10} (no log)")
        continue
    rows = [json.loads(l) for l in log.open()]
    vals = [r for r in rows if "val_onset" in r]
    for r in vals:
        print(f"{tag:<10} {r['step']:>6} {r['val_total']:>10.4f} "
              f"{r['val_onset']:>10.4f}")
    if vals:
        summary[tag] = vals[-1]

print("\n--- final val_onset, against the control ---")
base = summary.get("boost1", {}).get("val_onset")
for tag, r in summary.items():
    delta = "" if base is None else f"  ({r['val_onset'] - base:+.4f} vs control)"
    print(f"{tag:<10} {r['val_onset']:.4f}{delta}")

## Reading the result

**`val_onset` is a stability check, not evidence the idea works.** 1,500 steps
cannot show an F1 gain; the deficit this targets is 67% of missed notes, and
moving it takes a real run.

| observation | reading |
|---|---|
| a boosted arm's `val_onset` is far WORSE than the control | the weighting destabilises training — **stop**, no full run |
| all arms sit within noise of the control | mechanism trains cleanly — **proceed**, pick the middle boost |
| a boosted arm is clearly BETTER | promising, but do not over-read 1,500 steps |

Per-step training noise in this project is σ ≈ 0.0111 on the total — **larger
than most real movement**. Treat anything under ~0.01 as noise, and read the
three boosts as a trend rather than ranking them.

## If it passes

The full run mirrors `calibration_run.ipynb`, with `--soft-onset-boost` set and
**in this order afterwards**:

```bash
# 1. RECALIBRATE FIRST. A retrained onset head invalidates the 0.6 threshold,
#    and scoring through a stale one misattributes a decode artifact to the
#    weights -- the mistake Phase 19 undid and Phase 23 nearly repeated.
set PTIFY_CHECKPOINT=C:\path\to\ptify-note-pedal.pth
python -m tools.calibrate_thresholds --audio-dir recordings/maps_paired \
    --engine ptify --limit 6

# 2. Score at the RECALIBRATED threshold against the controlled baseline,
#    benchmarks/real/maps-paired-ptify16b-at060.json (0.8502).
#    Diff onset_p as well as onset_f1: this change pushes toward inventing
#    notes, and 16b's entire published gain was a 37% cut in exactly those.

# 3. Re-run the Phase 27 diagnosis. It is the only measurement that answers
#    the actual question -- did the pp miss rate fall?
python -m tools.recall_diagnosis --audio-dir recordings/maestro_test12 \
    --engine ptify --json benchmarks/recall-diagnosis-phase28.json
```

**The success criterion is the pp miss rate falling toward the mf rate while
`onset_p` holds.** Recall alone is not success: any change that invents more
notes raises recall, and that is precisely how Phase 23 lost.